# Example: A Standard-Listed SPY Protective Collar
This example builds a collar from ordinary listed `SPY` options: long 100 shares, long one put, and short one call with the same expiration. No FLEX options are required. TJUL appears only at the end as a defined-outcome product comparison.

> __Tasks__
> 1. Translate annualized log growth $g$ into terminal `SPY` price using $r=(\Delta t)g$.
> 2. Size the shares and standard option contracts consistently.
> 3. Compute floor, cap, breakeven, maximum loss, and maximum profit using executable option quotes.
> 4. Compare the retail-replicable collar with TJUL without attempting to trade FLEX options.
___

In [ ]:
include(joinpath(@__DIR__, "Include.jl"));

## 1. Construct the standard-listed order
A standard equity or ETF option contract ordinarily represents 100 shares. We therefore match 100 `SPY` shares with one put and one call. The quotes below are deliberately illustrative—not current market data. Both options use the same 90-day expiration.

For a conservative opening estimate, buy the put at its ask and sell the call at its bid. A broker may fill a multi-leg limit order inside those individual quotes, but the mid price is not guaranteed.


In [ ]:
ticker = "SPY";
S₀ = 600.0;             # illustrative share purchase price (USD/share)
m = 100;                # ordinary contract multiplier (shares/contract)
n_contracts = 1;
n_shares = 100;
expiration_days = 90;
Δt = expiration_days / 365; # years

K_put = 540.0;          # 10% below S₀
put_bid = 11.00;
put_ask = 11.50;        # executable-side estimate for a purchase
K_call = 660.0;         # 10% above S₀
call_bid = 10.00;       # executable-side estimate for a sale
call_ask = 10.50;

@assert n_shares == m * n_contracts "Match 100 shares to each standard contract";
@assert K_put < S₀ < K_call "Use K_put < S₀ < K_call for a conventional collar";

net_option_debit_per_share = put_ask - call_bid;
initial_position_cost = n_shares * S₀ + m * n_contracts * net_option_debit_per_share;

The opening ticket is:

* buy 100 shares of `SPY`;
* buy to open one $K_P=540$ put; and
* sell to open one $K_C=660$ call with the same expiration.

If the shares are already owned, only the two option legs are opened. If the broker supports a stock-plus-options combination order, all three legs can be submitted together with one net limit. Check the option approval level, buying-power treatment, deliverable, expiration, and ex-dividend calendar before entering an order.
___

## 2. Map growth into the expiration payoff
The course convention is $r=(\Delta t)g$. Thus

$$S_T=S_0\exp((\Delta t)g),\qquad R=\frac{S_T-S_0}{S_0}=\exp((\Delta t)g)-1.$$

The option premiums are initial cash flows. They shift profit but do not alter the terminal-value floor $K_P$ or cap $K_C$.


In [ ]:
g = range(log(0.70) / Δt, log(1.30) / Δt, length=701) |> collect;
r = Δt .* g;
S_T = S₀ .* exp.(r);
R_spy = exp.(r) .- 1.0;

put_payoff_per_share = max.(K_put .- S_T, 0.0);
call_payoff_per_share = max.(S_T .- K_call, 0.0);
terminal_collar_value = n_shares .* S_T .+
    m * n_contracts .* put_payoff_per_share .-
    m * n_contracts .* call_payoff_per_share;
collar_profit = terminal_collar_value .- initial_position_cost;
stock_profit = n_shares .* (S_T .- S₀);

collar_df = DataFrame(g=g, r=r, S_T=S_T, R_spy=R_spy,
    stock_profit=stock_profit, collar_profit=collar_profit);

In [ ]:
floor_value = m * n_contracts * K_put;
cap_value = m * n_contracts * K_call;
maximum_loss = initial_position_cost - floor_value;
maximum_profit = cap_value - initial_position_cost;
breakeven_price = initial_position_cost / n_shares;

risk_summary = DataFrame(
    quantity=["Initial position cost", "Net option debit", "Floor value",
        "Cap value", "Breakeven share price", "Maximum loss", "Maximum profit"],
    value=[initial_position_cost, m * n_contracts * net_option_debit_per_share,
        floor_value, cap_value, breakeven_price, maximum_loss, maximum_profit],
    units=["USD/position", "USD/position", "USD/position",
        "USD/position", "USD/share", "USD/position", "USD/position"]
);
pretty_table(risk_summary);

In [ ]:
plot(S_T, stock_profit, lw=3, label="100 SPY shares", c=:gray55,
    bg="floralwhite", background_color_outside="white", framestyle=:box)
plot!(S_T, collar_profit, lw=4, label="Standard-listed protective collar", c=:navyblue)
hline!([0.0], lw=1.5, ls=:dash, label="", c=:black)
vline!([K_put, K_call], lw=1.5, ls=:dot,
    label=["Put strike" "Call strike"], c=[:red :green4])
xlabel!("SPY price at option expiration (USD/share)")
ylabel!("Position profit (USD)")

## 3. Strike selection with a standard option chain
Standard options restrict us to the listed strikes and expirations. The table below illustrates the decision. A higher put strike improves the floor; a lower call strike generally finances more of that protection but caps upside sooner. Always calculate the opening debit using put ask minus call bid before considering a midpoint or negotiated multi-leg limit.


In [ ]:
collar_candidates = DataFrame(
    K_put=[530.0, 540.0, 550.0, 560.0, 570.0],
    put_ask=[8.40, 11.50, 15.40, 20.30, 26.60],
    K_call=[670.0, 660.0, 650.0, 640.0, 630.0],
    call_bid=[7.10, 10.00, 13.80, 18.50, 24.90]
);
collar_candidates.net_debit = collar_candidates.put_ask .- collar_candidates.call_bid;
collar_candidates.floor_price_change = collar_candidates.K_put ./ S₀ .- 1.0;
collar_candidates.cap_price_change = collar_candidates.K_call ./ S₀ .- 1.0;
pretty_table(collar_candidates);

## 4. Expiration and early-assignment mechanics

* If $S_T<K_P$, exercising or assigning the put sells the matched shares at $K_P$.
* If $K_P\leq S_T\leq K_C$, both options normally expire without exercise and the shares remain.
* If $S_T>K_C$, call assignment sells the shares at $K_C$.
* Standard equity/ETF options are generally American-style, so assignment can occur before expiration. Short-call assignment risk is especially relevant around an ex-dividend date when remaining extrinsic value is small.
* Corporate actions can adjust the deliverable away from the ordinary 100 shares; verify the contract specification rather than relying only on the ticker and strike.

A retail implementation should use limit orders, avoid leaving an unintended naked option while adjusting legs, and decide in advance whether to close, roll, exercise, or accept assignment.
___

## 5. Compare with TJUL—without trading FLEX options
TJUL uses FLEX options internally to target a particular two-year cap and buffer. A student or retail investor does not need FLEX access to analyze or compare the ETF: TJUL shares themselves are exchange-traded, while our self-managed alternative uses only ordinary listed `SPY` options.

The simplified complete-period TJUL map below is a product comparison, not a replication of its live FLEX holdings or a mid-period NAV model. Parameters were checked on August 4, 2026 and must be refreshed before class.


In [ ]:
tjul_cap_gross = 0.1361;
tjul_buffer_gross = 1.00;
management_fee = 0.0079;
outcome_years = 2.0;

function defined_outcome(R_spy; cap, buffer)
    R_spy >= 0.0 && return min(R_spy, cap)
    return min(R_spy + buffer, 0.0)
end

R_spy_grid = range(-1.0, 0.40, length=701) |> collect;
R_tjul_gross = defined_outcome.(R_spy_grid;
    cap=tjul_cap_gross, buffer=tjul_buffer_gross);
fee_over_period = management_fee * outcome_years;
R_tjul_fee_illustration = R_tjul_gross .- fee_over_period;

plot(100 .* R_spy_grid, 100 .* R_spy_grid, lw=3, label="SPY price change", c=:gray55,
    bg="floralwhite", background_color_outside="white", framestyle=:box)
plot!(100 .* R_spy_grid, 100 .* R_tjul_gross, lw=4,
    label="TJUL gross outcome abstraction", c=:navyblue)
plot!(100 .* R_spy_grid, 100 .* R_tjul_fee_illustration, lw=3, ls=:dash,
    label="After management-fee illustration", c=:red3)
hline!([0.0, 100 * tjul_cap_gross], lw=1.2, ls=:dot,
    label=["" "Gross cap"], c=[:black :green4])
xlabel!("Cumulative SPY price change (%)")
ylabel!("Complete-period outcome (%)")

A 540/660 standard collar protects only below its 540 put strike; it still participates in the first 10% decline. TJUL's advertised starting full buffer is economically different. Matching it more closely with standard options would require a put nearer $S_0$, would depend on available listed strikes and expirations, and would generally require a different call cap or a net debit.

TJUL also references `SPY` price return, while direct `SPY` ownership may receive distributions. Entry date, remaining cap/buffer, fees, bid--ask spread, taxes, and holding through the outcome-period end all affect the comparison. Read the current [fund page](https://www.innovatoretfs.com/etf/?ticker=TJUL), [prospectus](https://www.innovatoretfs.com/pdf/tjul_prospectus.pdf), and [factsheet](https://www.innovatoretfs.com/pdf/tjul_factsheet.pdf).

__Educational use and risk notice.__ This illustrative example is not current market data, investment advice, or an order recommendation.
___

## Optional advanced extension
Replace the illustrative quotes with a dated standard `SPY` option chain. Search feasible put/call pairs using executable bid and ask prices, impose a maximum loss or minimum floor, and identify the pair with the greatest call strike subject to a maximum net debit. The optional Wheel notebooks remain in [`advanced/`](advanced/).
